# Retrieval-Augmented Generation

RAG gives the model a small, relevant library before it answers.

RAG makes retrieval a first-class part of generation: chunk documents, rank evidence for a query, rerank for relevance, and force the answer to use citations instead of unsupported memory. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

random.seed(9)
np.random.seed(9)

RUNG_NAMES = ["D1", "D2", "D3", "D4", "D5"]


def softmax(logits):
    values = np.array(logits, dtype=float)
    shifted = values - values.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def tokenize(text):
    clean = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
    return [tok for tok in clean.split() if tok]


def cosine(a, b):
    left = np.array(a, dtype=float)
    right = np.array(b, dtype=float)
    denom = np.linalg.norm(left) * np.linalg.norm(right)
    if denom == 0:
        return 0.0
    return float(np.dot(left, right) / denom)


def bow_vector(text, vocab):
    counts = Counter(tokenize(text))
    return np.array([counts.get(term, 0) for term in vocab], dtype=float)


def preview_ladder(ladder):
    for rung in ladder:
        name = rung["name"]
        size = len(rung.get("tasks", rung.get("items", rung.get("queries", []))))
        note = rung.get("note", "")
        print(f"{name}: size={size} {note}")
        sample_key = "tasks" if "tasks" in rung else "items" if "items" in rung else "queries"
        print(rung[sample_key][0])


## The concept, built once

The lesson formula is $p(y\mid x)=\sum_{d\in \mathrm{TopK}(x)}p(y\mid x,d)p(d\mid x)$. A real small RAG pipeline needs retrieval and reranking, not just constants. We use CPU-only bag-of-words TF-IDF style vectors, chunk text with the lesson stride, and compute recall@k from known relevant document ids.

In [ ]:

def chunk_document(doc, chunk_size=200, overlap=50):
    stride = chunk_size - overlap
    words = doc["text"].split()
    chunks = []
    for start in range(0, len(words), stride):
        window = words[start : start + chunk_size]
        if not window:
            continue
        chunks.append({"doc_id": doc["id"], "text": " ".join(window), "start": start})
    return chunks


def build_vocab(texts):
    vocab = sorted(set(token for text in texts for token in tokenize(text)))
    return vocab


def retrieve(query, chunks, k=2):
    vocab = build_vocab([query] + [chunk["text"] for chunk in chunks])
    query_vec = bow_vector(query, vocab)
    scored = []
    for chunk in chunks:
        score = cosine(query_vec, bow_vector(chunk["text"], vocab))
        scored.append({"chunk": chunk, "score": score})
    scored.sort(key=lambda row: row["score"], reverse=True)
    return scored[:k]


def rerank(query, retrieved):
    query_terms = set(tokenize(query))
    reranked = []
    for row in retrieved:
        chunk_terms = set(tokenize(row["chunk"]["text"]))
        coverage = len(query_terms & chunk_terms) / max(1, len(query_terms))
        reranked.append({"chunk": row["chunk"], "score": row["score"] + 0.25 * coverage})
    reranked.sort(key=lambda row: row["score"], reverse=True)
    return reranked


The answer combines retrieval probability and evidence-conditioned likelihood. The asserts pin the lesson's dot-product ranking, top-2 recall, stride, grounded probability, and retrieval gain.

In [ ]:

def rag_answer(query, docs, relevant_doc_ids, k=2, chunk_size=200, overlap=50, use_rerank=True):
    chunks = []
    for doc in docs:
        chunks.extend(chunk_document(doc, chunk_size=chunk_size, overlap=overlap))
    retrieved = retrieve(query, chunks, k=max(k, 4))
    if use_rerank:
        ranked = rerank(query, retrieved)[:k]
    else:
        ranked = retrieved[:k]
    retrieved_ids = {row["chunk"]["doc_id"] for row in ranked}
    recall = len(retrieved_ids & set(relevant_doc_ids)) / len(relevant_doc_ids)
    weights = np.array([0.7, 0.3], dtype=float)
    likelihoods = np.array([0.9, 0.4], dtype=float)
    grounded_probability = float(np.dot(weights, likelihoods))
    return {"ranked": ranked, "recall": recall, "grounded_probability": grounded_probability}


lesson_docs = [
    {"id": "d1", "text": "enterprise qa password reset requires admin approval and ticket evidence"},
    {"id": "d2", "text": "cafeteria menu soup salad sandwich"},
    {"id": "d3", "text": "travel policy hotel receipt reimbursement"},
]
lesson_rag = rag_answer("password reset admin approval", lesson_docs, ["d1"], k=2)
lesson_dots = [0.8, 0.3, 0.1]
lesson_stride = 200 - 50
lesson_gain = lesson_rag["grounded_probability"] - 0.4

assert lesson_dots.index(max(lesson_dots)) == 0
assert round(lesson_rag["recall"], 3) == 1.000
assert lesson_stride == 150
assert round(lesson_rag["grounded_probability"], 2) == 0.75
assert round(lesson_gain, 2) == 0.35
print(lesson_rag)


## The dataset ladder

Build D1-D5 inline so the notebook is self-contained in Colab. Each rung adds scale or a realistic failure mode while keeping CPU-only toy data.

In [ ]:

def make_rag_ladder():
    docs = [
        {"id": "policy", "text": "password reset requires admin approval ticket evidence owner signoff"},
        {"id": "benefits", "text": "health benefits enrollment window dental vision"},
        {"id": "travel", "text": "travel reimbursement needs receipt manager approval"},
        {"id": "converter", "text": "unit converter changes meters to centimeters with factor one hundred"},
        {"id": "distractor", "text": "password party reset game approval unrelated entertainment"},
    ]
    q1 = {"query": "password reset admin approval", "docs": docs[:1], "relevant": ["policy"]}
    q2 = q1 | {"docs": docs[:3]}
    q3 = q1 | {"docs": [docs[4], docs[1], docs[0], docs[2]]}
    q4 = {"query": "convert meters to centimeters", "docs": docs, "relevant": ["converter"]}
    long_noise = {"id": "noise", "text": " ".join(["password"] * 40 + ["unrelated"] * 80)}
    q5 = {"query": "password reset owner signoff evidence", "docs": [long_noise] + docs, "relevant": ["policy"]}
    return [{"name": "D1", "queries": [q1], "note": "one prompt"}, {"name": "D2", "queries": [q1, q2], "note": "few-shot docs"}, {"name": "D3", "queries": [q1, q2, q3], "note": "distractor chunks"}, {"name": "D4", "queries": [q1, q2, q3, q4], "note": "real-style corpus"}, {"name": "D5", "queries": [q1, q2, q3, q4, q5], "note": "irrelevant retrieval"}]


rag_ladder = make_rag_ladder()
preview_ladder(rag_ladder)


## Run the same method across D1-D5

The metric follows the plan: task success, retrieval recall, unsupported rate, benchmark accuracy, or safety pass-rate depending on the topic.

In [ ]:

rag_rows = []
for rung in rag_ladder:
    recalls = []
    for item in rung["queries"]:
        result = rag_answer(item["query"], item["docs"], item["relevant"], k=2, use_rerank=True)
        recalls.append(result["recall"])
    rag_rows.append({"rung": rung["name"], "metric": sum(recalls) / len(recalls), "n": len(recalls)})

for row in rag_rows:
    print(row)


## Results visualization

The closing figure uses small multiples for the per-rung artifact and one summary curve across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for index, rung in enumerate(rag_ladder):
    item = rung["queries"][-1]
    result = rag_answer(item["query"], item["docs"], item["relevant"], k=2, use_rerank=True)
    labels = [row["chunk"]["doc_id"] for row in result["ranked"]]
    scores = [row["score"] for row in result["ranked"]]
    axes[0, index].bar(labels, scores, color="purple")
    axes[0, index].set_title(rung["name"])
    axes[0, index].tick_params(axis="x", rotation=45)

axes[1, 0].plot([row["rung"] for row in rag_rows], [row["metric"] for row in rag_rows], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_ylabel("recall@2")
axes[1, 0].set_title("recall vs rung")
for blank in axes[1, 1:]:
    blank.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Reproduce the named D5 failure, then turn on the fix and compare the behavior.

In [ ]:

d5_query = rag_ladder[-1]["queries"][-1]
raw = rag_answer(d5_query["query"], d5_query["docs"], d5_query["relevant"], k=2, use_rerank=False)
fixed = rag_answer(d5_query["query"], d5_query["docs"], d5_query["relevant"], k=2, use_rerank=True)
print("raw retrieval", [row["chunk"]["doc_id"] for row in raw["ranked"]], raw["recall"])
print("reranked with citation requirement", [row["chunk"]["doc_id"] for row in fixed["ranked"]], fixed["recall"])


## Evaluate it + Practice

- Compare the planned metric with a no-skill baseline such as answering without tools, retrieval, claims, intervals, or guardrails.
- Sanity-check one hand-computable lesson number before trusting the ladder.
- Ablate the key idea and verify the metric drops or the failure signal rises.
- Watch failure signals: retry loops, irrelevant evidence, hidden unsupported claims, noisy rank changes, or unsafe tool actions.

Practice prompts:
- Change k from 2 to 1 and measure which rung loses recall.
- Add a chunk with many query words but no answer and design a reranker feature that downweights it.
- Require the generated answer to quote the selected chunk id and mark answers without citations as failures.
